In [9]:
%set_env TOKENIZERS_PARALLELISM=false
import os
import numpy as np
import torch

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

from esm.utils.structure.protein_chain import ProteinChain
from esm.utils.constants.esm3 import SEQUENCE_VOCAB
from esm.models.esm3 import ESM3
from esm.sdk.api import (
    ESMProtein,
    GenerationConfig,
)
model =  ESM3.from_pretrained("esm3_sm_open_v1", device=torch.device(DEVICE)).eval()


env: TOKENIZERS_PARALLELISM=false


Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

In [10]:
from esm.pretrained import (
    ESM3_function_decoder_v0,
    ESM3_sm_open_v0,
    ESM3_structure_decoder_v0,
)
from esm.tokenization.function_tokenizer import (
    InterProQuantizedTokenizer as EsmFunctionTokenizer,
)
from esm.tokenization.sequence_tokenizer import (
    EsmSequenceTokenizer,
)
from esm.utils.constants.esm3 import (
    SEQUENCE_MASK_TOKEN,
)
from esm.utils.structure.protein_chain import ProteinChain
from esm.utils.types import FunctionAnnotation

tokenizer = EsmSequenceTokenizer()
function_tokenizer = EsmFunctionTokenizer()

In [15]:
torch._dynamo.config.suppress_errors = True
decoder = ESM3_structure_decoder_v0(DEVICE)
function_decoder = ESM3_function_decoder_v0(DEVICE)
function_tokenizer = EsmFunctionTokenizer()

In [ ]:
from go_ml.train_utils import get_enzyme_df
enzyme_df = get_enzyme_df()
print(enzyme_df.iloc[0].Sequence)

In [13]:
# PDB 1UTN
sequence = enzyme_df.iloc[0].Sequence
tokens = tokenizer.encode(sequence)
sequence_tokens = torch.tensor(tokens, dtype=torch.int64)
sequence_tokens = sequence_tokens.cuda().unsqueeze(0)
output = model.forward(sequence_tokens=sequence_tokens, function_tokens=None)

In [17]:
import torch.nn.functional as F

p_none_threshold = 0.05
log_p = F.log_softmax(output.function_logits[:, 1:-1, :], dim=3).squeeze(0)

# Choose which positions have no predicted function.
log_p_nones = log_p[:, :, function_tokenizer.vocab_to_index["<none>"]]
p_none = torch.exp(log_p_nones).mean(dim=1)  # "Ensemble of <none> predictions"
where_none = p_none > p_none_threshold  # (length,)

log_p[~where_none, :, function_tokenizer.vocab_to_index["<none>"]] = -torch.inf
function_token_ids = torch.argmax(log_p, dim=2)
function_token_ids[where_none, :] = function_tokenizer.vocab_to_index["<none>"]

predicted_function = function_decoder.decode(
    function_token_ids,import torch.nn.functional as F

p_none_threshold = 0.05
log_p = F.log_softmax(output.function_logits[:, 1:-1, :], dim=3).squeeze(0)

# Choose which positions have no predicted function.
log_p_nones = log_p[:, :, function_tokenizer.vocab_to_index["<none>"]]
p_none = torch.exp(log_p_nones).mean(dim=1)  # "Ensemble of <none> predictions"
where_none = p_none > p_none_threshold  # (length,)

log_p[~where_none, :, function_tokenizer.vocab_to_index["<none>"]] = -torch.inf
function_token_ids = torch.argmax(log_p, dim=2)
function_token_ids[where_none, :] = function_tokenizer.vocab_to_index["<none>"]

predicted_function = function_decoder.decode(
    function_token_ids,
    tokenizer=function_tokenizer,
    annotation_threshold=0.1,
    annotation_min_length=5,
    annotation_gap_merge_max=3,
)
    tokenizer=function_tokenizer,
    annotation_threshold=0.1,
    annotation_min_length=5,
    annotation_gap_merge_max=3,
)

In [19]:
print(predicted_function.keys())
print("function prediction:")
print(predicted_function["interpro_preds"].nonzero())
print(predicted_function["function_keywords"])

dict_keys(['keyword_logits', 'keyword_tfidf', 'interpro_logits', 'interpro_preds', 'interpro_annotations', 'function_keywords'])
function prediction:
tensor([[    0, 16367],
        [    1, 16367],
        [    2, 16367],
        ...,
        [  361, 20575],
        [  362, 16367],
        [  363, 16367]], device='cuda:0')
[FunctionAnnotation(label='enolase', start=1, end=101), FunctionAnnotation(label='enolase', start=103, end=105), FunctionAnnotation(label='enolase', start=107, end=364), FunctionAnnotation(label='enolase like', start=1, end=242), FunctionAnnotation(label='enolase like', start=246, end=246), FunctionAnnotation(label='enolase like', start=257, end=257), FunctionAnnotation(label='enolase like', start=259, end=259), FunctionAnnotation(label='enolase like', start=262, end=262), FunctionAnnotation(label='enolase like', start=266, end=266), FunctionAnnotation(label='enolase like', start=273, end=273), FunctionAnnotation(label='enolase like', start=277, end=277), FunctionAnn